# 07 · Saving, loading, and inference
**Goals:** save learned weights and the information needed to recreate preprocessing and architecture. This notebook trains its own small model, so it does not depend on another notebook. Only load files from trusted sources.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Small examples run efficiently on a laptop.
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."
frame = pd.read_csv(data_path)
features = ["feature_1", "feature_2", "feature_3"]
train = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"]
test = frame[frame["split"] == "test"]
scaler = StandardScaler().fit(train[features])  # Never fit on validation or test.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)
x_test, y_test = tensors(test)


In [ ]:
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)
model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()  # Inputs are raw logits; labels are integer indices.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            count += len(y_batch)
    return total_loss / count, correct / count

history = []
for epoch in range(1, 21):
    model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
    train_loss, train_accuracy = evaluate(model, DataLoader(TensorDataset(x_train, y_train), batch_size=64))
    val_loss, val_accuracy = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss, train_accuracy, val_accuracy))
print(f"Epoch {epoch}: train loss={train_loss:.3f}, validation loss={val_loss:.3f}, validation accuracy={val_accuracy:.3f}")


Save a state dictionary together with simple metadata. Here the preprocessing is a standardization with means and scales; a real project may require additional transformations, categorical encoders, and a label mapping. The architecture definition must be recreated before loading its weights.

In [ ]:
artifact_dir = root / "artifacts"
artifact_dir.mkdir(exist_ok=True)
checkpoint_path = artifact_dir / "classifier.pt"
checkpoint = {
    "model_state": model.state_dict(),
    "n_features": len(features), "hidden_units": 16, "n_classes": 2,
    "feature_names": features, "class_names": ["class 0", "class 1"],
    "mean": torch.tensor(scaler.mean_, dtype=torch.float32),
    "scale": torch.tensor(scaler.scale_, dtype=torch.float32),
}
torch.save(checkpoint, checkpoint_path)
print("saved:", checkpoint_path)


In [ ]:
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = nn.Sequential(nn.Linear(saved["n_features"], saved["hidden_units"]),
                         nn.ReLU(), nn.Linear(saved["hidden_units"], saved["n_classes"]))
restored.load_state_dict(saved["model_state"])
restored.eval()
# New raw data must use the same feature order and training-fitted transformation.
new_rows = pd.DataFrame([[0.5, -1.0, 0.2], [-1.0, 1.5, 0.0]], columns=saved["feature_names"])
raw = torch.tensor(new_rows[saved["feature_names"]].to_numpy(), dtype=torch.float32)
standardized = (raw - saved["mean"]) / saved["scale"]
with torch.no_grad():
    probabilities = torch.softmax(restored(standardized), dim=1)
    predictions = probabilities.argmax(dim=1)
print("probabilities:", probabilities)
print("predicted labels:", [saved["class_names"][i] for i in predictions.tolist()])
with torch.no_grad():
    torch.testing.assert_close(model(x_val), restored(x_val))
print("Reloaded model matches the original on validation inputs.")


A training-resume checkpoint also needs optimizer state, epoch, and any scheduler state; for exact reproducibility it may need random-number-generator state and sampler details. An inference checkpoint alone is not enough to resume Adam exactly. `weights_only=True` restricts deserialization, but use trusted files. The checkpoint is written to the Git-ignored `artifacts/` directory.

## Try it
1. Change a raw row and interpret the predicted class and probabilities.
2. Compare predictions with and without standardization. Why does saving the scaler matter?
3. Save and reload `optimizer.state_dict()` with an epoch number for a resumable training checkpoint.
4. Add a class-label mapping for a dataset whose labels are strings.